***CHECKING IF THE BUILT MLP WORKS***

In [1]:
from protgen.models.mlp import MLP

In [2]:
import torch
from torch import nn
import torch.nn.functional as F

In [3]:
import torch.optim as optim

In [ ]:
import matplotlib.pyplot as plt

Now that the mlp.py contains an actual Architecture we will now test if it is working. Initial approach uses 128 hidden units. Each embedding is 24 long

Adding same structure used in the Introduction:

In [4]:
from protgen.data import ProteinDataset
from torch.utils.data import DataLoader

train_data = ProteinDataset(path_to_data="../data/train.csv")
val_data = ProteinDataset(path_to_data="../data/val.csv")
test_data = ProteinDataset(path_to_data="../data/test.csv")

print(f"Data class: {train_data}")

# Get tensors
X_train = train_data.tokens
y_train_labels = train_data.labels          # classification task
y_train_shifted = train_data.tokens_shifted # generative task

print(f"# Train data: \n Tokens shape: {X_train.shape}\n Labels (shape): {y_train_labels.shape} \n Shifted shape: {y_train_shifted.shape}")


# Inddexing
x, x_shifted, y, length = train_data[0]
print(f"# Sample 0: \nTokens: \n {x}\n Shifted: \n {x_shifted}\n Label: {y}\n Length: {length}")

# DataLoader
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

Data class: ProteinDataset(num_samples=16020, vocab_size=23, num_classes=3, max_length=129)
# Train data: 
 Tokens shape: torch.Size([16020, 129])
 Labels (shape): torch.Size([16020]) 
 Shifted shape: torch.Size([16020, 129])
# Sample 0: 
Tokens: 
 tensor([ 1, 13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,
         3, 19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6,
        22, 16,  8, 11, 19, 10, 14, 12, 11,  9, 18,  3, 11, 18, 10, 13,  5, 10,
        13, 18, 12, 18, 10, 17, 15,  8, 19, 11,  7, 22, 10, 11,  3,  6,  8, 10,
         5,  6, 20, 16,  3, 10, 16, 12, 10, 16,  5, 22, 12, 22, 11,  6, 11, 12,
        10, 14, 22,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0])
 Shifted: 
 tensor([13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,  3,
        19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6, 22

In [5]:
val_loader = DataLoader(val_data, batch_size=32, shuffle=True)

In [6]:
mlp = MLP()


In [7]:
print(mlp)

MLP(
  (embedding): Embedding(23, 24, padding_idx=0)
  (net): Sequential(
    (0): Linear(in_features=72, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=23, bias=True)
  )
)


In [8]:
print(list(mlp.named_parameters()))

[('embedding.weight', Parameter containing:
tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 3.6922e-01,  1.1691e+00, -4.9908e-01,  1.3296e+00, -2.7892e-01,
          8.2929e-01,  4.4105e-01,  6.7290e-01,  3.2369e-01,  2.3259e-01,
          1.9330e+00,  3.9602e-01, -1.2751e-01,  9.3326e-01, -6.8214e-01,
         -3.9475e-01,  2.3331e-01, -3.7765e-01,  1.4750e+00,  5.0521e-01,
          2.5990e-01,  8.2951e-02, -5.9259e-01, -7.9033e-01],
        [ 3.6948e-01, -5.7663e-01, -2.5121e-01, -1.4262e+00,  3.0310e-01,
         -6.7156e-01, -3.3400e-01,  4.4713e-01,  1.6549e-01,  7.7800e-01,
         -1.0554e+00, -2.2278e+00,  1.9047e+00,  5.6896e-01, -2.7210e+00,
          1.4064e+

In [9]:
train_data.df

,sequence,family
0,MIKKQFNIIKNTGFARPATLLVKLSQRFTSDISLEYQGKTINLKHS...,HPr
1,MNKHELITRISEKSGMTKKDVESVLNGFMDEVTDALGSGNKVQLIG...,HU
2,MTQRFEDTFLIQNKLGLHARAASVLAELSAQFDANITLFQGSKEAE...,HPr
3,MNKTELILRIAEETGFSRKNVELGLNAAINLITKALTEEEKVQIVG...,HU
4,MVERKAVIKNEAGIHCRPTAVITQEAAKVDAVIMILSPGGTCQLGS...,HPr
...,...,...
16015,MTKADIINEIANETGVAKKDVTVVVESFMATIKDSLLEKRECVYLR...,HU
16016,MSRLSRQVTIVNQKGLHARASAKFVTFVSRLPEGLKVEVEKDGQCV...,HPr
16017,MNKSDLINHVAAETGLTKADSGFALEAVLEGITKSLRKGDTVTLTG...,HU
16018,MTLTKAELADLLFEKVGLNKREAKDMVEAFFEEIRNALERGEGVKL...,HU


In [10]:
val_data.df

,sequence,family
0,MNKADFVKHVAQQHQCTQIEAEKTIDMFTSSVIDAMGKGNEISLVG...,HU
1,MSERKATIASRVGLHARPASIFAAAVGELPIEVTIGLEADPVEDAM...,HPr
2,MNKSQLIDKIAAGAELSKADAGRALDAFIGAVTDALKEGDSVALVG...,HU
3,MNKSELVDKMAEKSGLSKKDSDKALNAMLESVEEALTSGDKVQLIG...,HU
4,MNKTELIDHIASKADISKAAAGRTLDALIGAVKTTLKKGGTVTLVG...,HU
...,...,...
1996,MDNRSRREAPEAVDTHGYYKAEIEVTHGFGLHARPSVIFTRLAKSF...,HPr
1997,MNKSELIDAIAASADIPKAVAGRALDAVIDSVTGALKAGDSVVLVG...,HU
1998,MSEHESAGRKLVSVDFEIFGYVQGVCFRMYTEREGLRLGLVGWVKN...,AcyP
1999,MNKSELIDAIAESADISKAAAGRALDAMTNTITGALKNDDQVTLIG...,HU


In [11]:
test_data.df

,sequence,family
0,MTKAEVIVAIADKTGVDKASVSTTLESFFEIVKDTLAQGDAVYVRG...,HU
1,MERVTAVARGRVQGVGYRYYVSECAHATGIHGWVKNEPDGTVRIVA...,AcyP
2,MPQSSRTVLIQNRLGLHARAAVKLVELAQSFEAVLTIQNQEGKEAT...,HPr
3,MKHLNLKIHGKVQGVFFREKTKQLAQELGLNGYVKNTSDNCVEVVV...,AcyP
4,MTDRTRAHVYVSGKVQGVYFRATTREEAREQGVDGWVRNLDDGRVE...,AcyP
...,...,...
1996,MNKSELIQAVAERSELTKKDAGSAIQAAFDIISETLEKGETVQLIG...,HU
1997,MANKAELIDSVASKTGLTKKDATSAVDAVFETIQDNLSEGNKVQLI...,HU
1998,MPKLERQVTICNKLGLHARAATKLVVLASEFDASITLIQGEKQASA...,HPr
1999,MAAASAGGPSDPPTPQQSSPTKAVRVVVKGRVQGVFFRDWTVETAR...,AcyP


***CHECK WINDOWING IS WORKING - add print(window_3s) in forward***

In [12]:
mlp(torch.tensor([[1, 2, 3, 4, 5]]))

tensor([[[-0.3775, -0.0029,  0.0933, -0.1262, -0.1283,  0.0323, -0.2032,
           0.1425,  0.1414, -0.1305, -0.0999, -0.1268, -0.0468, -0.0445,
           0.1028, -0.0693,  0.0525, -0.0026,  0.1840, -0.0211, -0.2735,
          -0.1331,  0.1363],
         [ 0.0931, -0.2948, -0.0173,  0.0141,  0.0610, -0.3654,  0.3237,
          -0.3723,  0.2621,  0.2728, -0.0168,  0.5276,  0.0069,  0.0582,
          -0.1793, -0.2753, -0.3533, -0.3660, -0.2451,  0.0726,  0.1830,
           0.5310, -0.2999],
         [-0.0374,  0.3215,  0.2661,  0.1119, -0.0716, -0.0666,  0.0633,
           0.3166,  0.0017,  0.6493,  0.4751,  0.3276,  0.2473, -0.1779,
          -0.0589, -0.6180, -0.2094, -0.0588,  0.1390, -0.4310, -0.0623,
          -0.1664,  0.0500],
         [ 0.0810,  0.1970, -0.0558,  0.4812, -0.3444,  0.0027,  0.0007,
           0.2778, -0.0850, -0.0467, -0.2347,  0.3632, -0.1093,  0.2379,
          -0.2209,  0.0661,  0.2879, -0.6195,  0.3497, -0.1238, -0.1634,
           0.3100,  0.3053],
        

***TRAINING***

In [13]:
loss_fn = nn.CrossEntropyLoss(ignore_index = -1)
optimizer = optim.SGD(mlp.parameters(), lr= 0.01)

In [14]:
# Test the forward pass with dummy data (from assignments!)
out = mlp(torch.randint(0,23,(32,129)))
print("Output shape:", out.size())
print(f"Output logits:\n{out.detach().cpu().numpy()}")
print(f"Output probabilities:\n{out.softmax(-1).detach().cpu().numpy()}")

Output shape: torch.Size([32, 129, 23])
Output logits:
[[[ 1.35921761e-02  2.28508078e-02  1.46211535e-01 ... -3.11123356e-02
    1.26131415e-01 -4.27131653e-02]
  [-2.19770297e-02 -1.30960524e-01 -2.62682408e-01 ... -1.21505901e-01
   -6.73541427e-03  2.40748897e-02]
  [-8.46688822e-02  3.07322852e-02 -3.13554257e-01 ...  1.40644506e-01
    3.30529988e-01  6.76075399e-01]
  ...
  [ 3.24901581e-01  2.34277546e-01 -1.48375496e-01 ...  2.16223836e-01
   -4.13995683e-02  1.39823854e-01]
  [ 4.03909713e-01  9.58307236e-02 -2.64021307e-01 ...  1.14115253e-01
   -2.04913855e-01  1.71285942e-02]
  [ 6.87584579e-02 -1.41992450e-01  4.16432977e-01 ... -2.38878474e-01
    3.03773910e-01  2.90573388e-01]]

 [[ 1.35921761e-02  2.28508078e-02  1.46211535e-01 ... -3.11123356e-02
    1.26131415e-01 -4.27131653e-02]
  [ 1.29065186e-01  1.79172665e-01 -1.86926857e-01 ... -5.52385449e-02
   -2.79664338e-01  1.05889492e-01]
  [-1.33031785e-01 -1.61029369e-01 -1.10704854e-01 ... -2.53849745e-01
    2.7653

In [15]:
print(len(train_loader))

501


In [16]:
num_epochs = 10
val_losses = []
train_losses = []

for epoch in range(num_epochs):
    
    ### Training
    mlp.train()
    total = 0
    for x, x_shifted, y, length in train_loader:
        optimizer.zero_grad()
        out = mlp(x)
        loss = loss_fn(out.transpose(1, 2), x_shifted)
        loss.backward()
        optimizer.step()
        total += loss.item()
    train_loss = total / len(train_loader)
    train_losses.append(train_loss)

    # validation
    mlp.eval()
    total = 0
    with torch.no_grad():
        for x, x_shifted, y, length in val_loader:
            out = mlp(x)
            loss = loss_fn(out.transpose(1, 2), x_shifted)
            total += loss.item()
    val_loss = total / len(val_loader)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs} | train {train_loss:.4f} | val {val_loss:.4f}")

plt.figure()
plt.plot(epoch, train_loss, 'r', epoch, val_loss, 'b')
plt.legend(['Train Loss','Validation Loss'])
plt.xlabel('Updates'), 
plt.ylabel('Loss')

Epoch 1/10 | train 2.9868 | val 2.8518
Epoch 2/10 | train 2.8114 | val 2.7572
Epoch 3/10 | train 2.7522 | val 2.7150
Epoch 4/10 | train 2.7212 | val 2.6893
Epoch 5/10 | train 2.7007 | val 2.6716
Epoch 6/10 | train 2.6859 | val 2.6578
Epoch 7/10 | train 2.6746 | val 2.6477
Epoch 8/10 | train 2.6654 | val 2.6392
Epoch 9/10 | train 2.6577 | val 2.6314
Epoch 10/10 | train 2.6509 | val 2.6258


NameError: name 'plt' is not defined